In [1]:
import os
import cv2
from deepface import DeepFace
from tqdm import tqdm
import csv

os.getcwd()

I0000 00:00:1785665846.848468  194827 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785665847.431156  194827 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785665849.892054  194827 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


'/media/tae/Tae_inHDD/Tae_ws/Ubuntu/Proj_Automatically_configure_infotainment_settings_by_estimating_age_and_gender/test'

In [2]:
# 1. 비디오 폴더 및 설정
videos_dir = "../videos"  # 테스트 비디오들이 저장된 폴더 경로 (필요 시 수정)
valid_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm')
frame_skip = 10  # 몇 프레임마다 AI 분석을 수행할지 설정 (1: 매 프레임 분석, 5: 5프레임마다 1번 분석으로 속도 향상)

# 출력 결과 저장 폴더 생성
output_dir = "./videos_test_result"
os.makedirs(output_dir, exist_ok=True)

if os.path.exists(videos_dir):
    video_files = sorted([f for f in os.listdir(videos_dir) if f.lower().endswith(valid_extensions)])
    
    if not video_files:
        print(f"❌ '{videos_dir}' 폴더에 분석할 비디오 파일이 없습니다.")
    else:
        print(f"📂 총 {len(video_files)}개의 비디오 파일을 발견했습니다. 분석을 시작합니다...\n")
        
        for v_idx, file_name in enumerate(video_files, 1):
            video_path = os.path.join(videos_dir, file_name)
            output_path = os.path.join(output_dir, f"output_{file_name}")
            
            print(f"==================================================")
            print(f"[{v_idx}/{len(video_files)}] 🎬 비디오 분석 중: {file_name}")
            print(f"==================================================")
            
            # 비디오 캡처 객체 생성
            cap = cv2.VideoCapture(video_path)
            if not cap.isOpened():
                print(f"❌ 비디오 파일을 열 수 없습니다: {file_name}\n")
                continue
                
            # 비디오 정보 가져오기
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            
            # 비디오 저장 객체 생성 (mp4v 코덱 사용)
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
            
            # [추가된 부분] CSV 파일 생성 및 헤더 작성
            csv_path = os.path.join(output_dir, f"output_{os.path.splitext(file_name)[0]}.csv")
            csv_file = open(csv_path, mode="w", newline="", encoding="utf-8")
            csv_writer = csv.writer(csv_file)
            csv_writer.writerow(["frame", "age", "gender", "score", "x", "y", "w", "h"])

            frame_count = 0
            current_results = []  # 최근 분석 결과 보존용
            
            # 진행 상황 표시줄 (tqdm)
            print(f"total frames = {total_frames}")
            pbar = tqdm(total=total_frames, desc=f"Processing {file_name}")
            
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                    
                frame_count += 1
                
                # frame_skip 간격마다 AI 분석 실행
                if frame_count % frame_skip == 1 or frame_skip == 1:
                    try:
                        # DeepFace 나이 및 성별 분석
                        results = DeepFace.analyze(
                            img_path = frame,
                            actions = ['age', 'gender'],
                            detector_backend = 'retinaface', # 'retinaface', 'opencv', 'mediapipe' 등
                            enforce_detection = False
                        )
                        current_results = results
                    except Exception as e:
                        current_results = []
                
                # 감지된 결과를 프레임 위에 시각화 (바운딩 박스 + 텍스트)
                for res in current_results:
                    region = res.get('region', {})
                    x, y, w, h = region.get('x', 0), region.get('y', 0), region.get('w', 0), region.get('h', 0)
                    
                    if w > 0 and h > 0:
                        age = res.get('age', 0)
                        gender = res.get('dominant_gender', 'Unknown')
                        gender_scores = res.get('gender', {})
                        score = gender_scores.get(gender, 0.0)
                        
                        # 1. 녹색 바운딩 박스 그리기
                        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

                        # 2. [추가된 부분] 현재 프레임의 정보 CSV에 한 줄씩 기록
                        csv_writer.writerow([frame_count, age, gender, round(score, 2), x, y, w, h])
                        
                        # 3. 결과 라벨 텍스트 생성 (예: "Man (98.5%) | 31y")
                        label = f"{gender} ({score:.1f}%) | {age}y"
                        
                        # 4. 텍스트 배경 라벨 그리기
                        (text_w, text_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                        cv2.rectangle(frame, (x, y - text_h - 10), (x + text_w, y), (0, 255, 0), cv2.FILLED)
                        cv2.putText(frame, label, (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
                
                # 결과 프레임 쓰기
                out.write(frame)
                pbar.update(1)
                
            pbar.close()
            cap.release()
            out.release()
            csv_file.close()
            print(f"✅ 분석 완료! 결과 비디오 저장 위치: {output_path}\n")
            print(f"✅ CSV 데이터 저장 위치: {csv_path}\n")
            
else:
    print(f"❌ '{videos_dir}' 경로를 찾을 수 없습니다.")

📂 총 8개의 비디오 파일을 발견했습니다. 분석을 시작합니다...

[1/8] 🎬 비디오 분석 중: test_1.mp4
total frames = 344


Processing test_1.mp4:   0%|          | 0/344 [00:00<?, ?it/s]E0000 00:00:1785665856.078317  194827 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
W0000 00:00:1785665860.184265  194827 cpu_allocator_impl.cc:82] Allocation of 22462464 exceeds 10% of free system memory.
W0000 00:00:1785665862.279621  194944 cpu_allocator_impl.cc:82] Allocation of 22462464 exceeds 10% of free system memory.
W0000 00:00:1785665862.314189  194944 cpu_allocator_impl.cc:82] Allocation of 22668240 exceeds 10% of free system memory.
W0000 00:00:1785665862.323556  194944 cpu_allocator_impl.cc:82] Allocation of 119799808 exceeds 10% of free system memory.
W0000 00:00:1785665862.492212  194944 cpu_allocator_impl.cc:82] Allocation of 120530944 exceeds 10% of free system memory.
Processing test_1.mp4: 100%|██████████| 344/344 [03:37<00:00,  1.58it/s]


✅ 분석 완료! 결과 비디오 저장 위치: ./videos_test_result/output_test_1.mp4

✅ CSV 데이터 저장 위치: ./videos_test_result/output_test_1.csv

[2/8] 🎬 비디오 분석 중: test_2.mp4
total frames = 683


Processing test_2.mp4: 100%|██████████| 683/683 [07:13<00:00,  1.58it/s]


✅ 분석 완료! 결과 비디오 저장 위치: ./videos_test_result/output_test_2.mp4

✅ CSV 데이터 저장 위치: ./videos_test_result/output_test_2.csv

[3/8] 🎬 비디오 분석 중: test_3.mp4
total frames = 372


...

✅ 분석 완료! 결과 비디오 저장 위치: ./videos_test_result/output_test_6.mp4

✅ CSV 데이터 저장 위치: ./videos_test_result/output_test_6.csv

[7/8] 🎬 비디오 분석 중: test_7.mp4
total frames = 330


Processing test_7.mp4: 100%|██████████| 330/330 [03:42<00:00,  1.48it/s]


✅ 분석 완료! 결과 비디오 저장 위치: ./videos_test_result/output_test_7.mp4

✅ CSV 데이터 저장 위치: ./videos_test_result/output_test_7.csv

[8/8] 🎬 비디오 분석 중: test_8.mp4
total frames = 1313


Processing test_8.mp4: 100%|██████████| 1313/1313 [15:50<00:00,  1.38it/s]

✅ 분석 완료! 결과 비디오 저장 위치: ./videos_test_result/output_test_8.mp4

✅ CSV 데이터 저장 위치: ./videos_test_result/output_test_8.csv

